In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
import torch 
from torch import nn,optim
from tqdm import tqdm
import torchinfo
from torchinfo import summary
import torchvision
from torch.utils.data import Dataset,DataLoader

In [2]:
data=pd.read_csv("../Data/final_data.csv")
data.head()

,channelname,category,subcategory,customerremarks,issuereportedat,issueresponded,surveyresponsedate,agentname,supervisor,manager,tenurebucket,agentshift,csatscore
0,Outcall,Product Queries,Life Insurance,NaN,2023-01-08 11:13:00,2023-01-08 11:47:00,2023-08-01,Richard Buchanan,Mason Gupta,Jennifer Nguyen,On Job Training,Morning,5
1,Outcall,Product Queries,Product Specific Information,NaN,2023-01-08 12:52:00,2023-01-08 12:54:00,2023-08-01,Vicki Collins,Dylan Kim,Michael Lee,>90,Morning,5
2,Inbound,Order Related,Installation/demo,NaN,2023-01-08 20:16:00,2023-01-08 20:38:00,2023-08-01,Duane Norman,Jackson Park,William Kim,On Job Training,Evening,5
3,Inbound,Returns,Reverse Pickup Enquiry,NaN,2023-01-08 20:56:00,2023-01-08 21:16:00,2023-08-01,Patrick Flores,Olivia Wang,John Smith,>90,Evening,5
4,Inbound,Cancellation,Not Needed,NaN,2023-01-08 10:30:00,2023-01-08 10:32:00,2023-08-01,Christopher Sanchez,Austin Johnson,Michael Lee,0-30,Morning,5


In [3]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 79738 entries, 0 to 79737
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   channelname         79738 non-null  str  
 1   category            79738 non-null  str  
 2   subcategory         79738 non-null  str  
 3   customerremarks     26701 non-null  str  
 4   issuereportedat     79738 non-null  str  
 5   issueresponded      79738 non-null  str  
 6   surveyresponsedate  79738 non-null  str  
 7   agentname           79738 non-null  str  
 8   supervisor          79738 non-null  str  
 9   manager             79738 non-null  str  
 10  tenurebucket        79738 non-null  str  
 11  agentshift          79738 non-null  str  
 12  csatscore           79738 non-null  int64
dtypes: int64(1), str(12)
memory usage: 19.1 MB


In [4]:
data.customerremarks=data.customerremarks.fillna("")
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 79738 entries, 0 to 79737
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   channelname         79738 non-null  str  
 1   category            79738 non-null  str  
 2   subcategory         79738 non-null  str  
 3   customerremarks     79738 non-null  str  
 4   issuereportedat     79738 non-null  str  
 5   issueresponded      79738 non-null  str  
 6   surveyresponsedate  79738 non-null  str  
 7   agentname           79738 non-null  str  
 8   supervisor          79738 non-null  str  
 9   manager             79738 non-null  str  
 10  tenurebucket        79738 non-null  str  
 11  agentshift          79738 non-null  str  
 12  csatscore           79738 non-null  int64
dtypes: int64(1), str(12)
memory usage: 19.1 MB


In [5]:
import textblob
from textblob import TextBlob

data["sentiment_score"] = data["customerremarks"].apply(
    lambda x: TextBlob(x).sentiment.polarity
)

In [6]:
data.head(20)

,channelname,category,subcategory,customerremarks,issuereportedat,issueresponded,surveyresponsedate,agentname,supervisor,manager,tenurebucket,agentshift,csatscore,sentiment_score
0,Outcall,Product Queries,Life Insurance,,2023-01-08 11:13:00,2023-01-08 11:47:00,2023-08-01,Richard Buchanan,Mason Gupta,Jennifer Nguyen,On Job Training,Morning,5,0.00
1,Outcall,Product Queries,Product Specific Information,,2023-01-08 12:52:00,2023-01-08 12:54:00,2023-08-01,Vicki Collins,Dylan Kim,Michael Lee,>90,Morning,5,0.00
2,Inbound,Order Related,Installation/demo,,2023-01-08 20:16:00,2023-01-08 20:38:00,2023-08-01,Duane Norman,Jackson Park,William Kim,On Job Training,Evening,5,0.00
3,Inbound,Returns,Reverse Pickup Enquiry,,2023-01-08 20:56:00,2023-01-08 21:16:00,2023-08-01,Patrick Flores,Olivia Wang,John Smith,>90,Evening,5,0.00
4,Inbound,Cancellation,Not Needed,,2023-01-08 10:30:00,2023-01-08 10:32:00,2023-08-01,Christopher Sanchez,Austin Johnson,Michael Lee,0-30,Morning,5,0.00
5,Email,Returns,Fraudulent User,,2023-01-08 15:13:00,2023-01-08 18:39:00,2023-08-01,Desiree Newton,Emma Park,John Smith,0-30,Morning,5,0.00
6,Outcall,Product Queries,Product Specific Information,,2023-01-08 15:31:00,2023-01-08 23:52:00,2023-08-01,Shannon Hicks,Aiden Patel,Olivia Tan,>90,Morning,5,0.00
7,Inbound,Returns,Exchange / Replacement,Very good,2023-01-08 16:17:00,2023-01-08 16:23:00,2023-08-01,Laura Smith,Evelyn Kimura,Jennifer Nguyen,On Job Training,Evening,5,0.91
8,Inbound,Returns,Missing,Shopzilla app and it's all coustomer care serv...,2023-01-08 21:03:00,2023-01-08 21:07:00,2023-08-01,David Smith,Nathan Patel,John Smith,>90,Split,5,0.91
9,Inbound,Shopzilla Related,General Enquiry,,2023-01-08 23:31:00,2023-01-08 23:36:00,2023-08-01,Tabitha Ayala,Amelia Tanaka,Michael Lee,31-60,Evening,5,0.00


In [7]:
df1=data.drop(columns="customerremarks")
df1.shape

(79738, 13)

In [8]:
df1.issuereportedat=df1.issuereportedat.astype("datetime64[ns]")
df1.issueresponded=df1.issueresponded.astype("datetime64[ns]")
df1.surveyresponsedate=df1.surveyresponsedate.astype("datetime64[ns]")

In [9]:
df1["issuemonth"]=df1.issuereportedat.dt.month.astype("float")
df1.head(2)

,channelname,category,subcategory,issuereportedat,issueresponded,surveyresponsedate,agentname,supervisor,manager,tenurebucket,agentshift,csatscore,sentiment_score,issuemonth
0,Outcall,Product Queries,Life Insurance,2023-01-08 11:13:00,2023-01-08 11:47:00,2023-08-01,Richard Buchanan,Mason Gupta,Jennifer Nguyen,On Job Training,Morning,5,0.0,1.0
1,Outcall,Product Queries,Product Specific Information,2023-01-08 12:52:00,2023-01-08 12:54:00,2023-08-01,Vicki Collins,Dylan Kim,Michael Lee,>90,Morning,5,0.0,1.0


In [10]:
df1["issuerespondedmonth"]=df1.issueresponded.dt.month.astype("float")
df1.head(2)

,channelname,category,subcategory,issuereportedat,issueresponded,surveyresponsedate,agentname,supervisor,manager,tenurebucket,agentshift,csatscore,sentiment_score,issuemonth,issuerespondedmonth
0,Outcall,Product Queries,Life Insurance,2023-01-08 11:13:00,2023-01-08 11:47:00,2023-08-01,Richard Buchanan,Mason Gupta,Jennifer Nguyen,On Job Training,Morning,5,0.0,1.0,1.0
1,Outcall,Product Queries,Product Specific Information,2023-01-08 12:52:00,2023-01-08 12:54:00,2023-08-01,Vicki Collins,Dylan Kim,Michael Lee,>90,Morning,5,0.0,1.0,1.0


In [11]:
df1["responsehours"]=(df1.issueresponded-df1.issuereportedat).dt.total_seconds()/3600

In [12]:
df1["responsehourscategory"]=pd.cut(df1.responsehours,bins=[0,6,12,18,24,3000],labels=["within 6","6-12","12-18","18-24","24+"])
df1.head(2)

,channelname,category,subcategory,issuereportedat,issueresponded,surveyresponsedate,agentname,supervisor,manager,tenurebucket,agentshift,csatscore,sentiment_score,issuemonth,issuerespondedmonth,responsehours,responsehourscategory
0,Outcall,Product Queries,Life Insurance,2023-01-08 11:13:00,2023-01-08 11:47:00,2023-08-01,Richard Buchanan,Mason Gupta,Jennifer Nguyen,On Job Training,Morning,5,0.0,1.0,1.0,0.566667,within 6
1,Outcall,Product Queries,Product Specific Information,2023-01-08 12:52:00,2023-01-08 12:54:00,2023-08-01,Vicki Collins,Dylan Kim,Michael Lee,>90,Morning,5,0.0,1.0,1.0,0.033333,within 6


In [13]:
df1.dtypes

channelname                         str
category                            str
subcategory                         str
issuereportedat          datetime64[ns]
issueresponded           datetime64[ns]
surveyresponsedate       datetime64[ns]
agentname                           str
supervisor                          str
manager                             str
tenurebucket                        str
agentshift                          str
csatscore                         int64
sentiment_score                 float64
issuemonth                      float64
issuerespondedmonth             float64
responsehours                   float64
responsehourscategory          category
dtype: object

In [14]:
x=df1.drop("csatscore",axis=1)
y=df1["csatscore"]
y.head(2),x.shape,y.shape

(0    5
 1    5
 Name: csatscore, dtype: int64,
 (79738, 16),
 (79738,))

In [15]:
numx=[col for col in x.columns if x[col].dtype=="float"]
catx=[col for col in x.columns if x[col].dtype!="float" ]
numx,catx

(['sentiment_score', 'issuemonth', 'issuerespondedmonth', 'responsehours'],
 ['channelname',
  'category',
  'subcategory',
  'issuereportedat',
  'issueresponded',
  'surveyresponsedate',
  'agentname',
  'supervisor',
  'manager',
  'tenurebucket',
  'agentshift',
  'responsehourscategory'])

In [16]:
from scipy.stats import chi2_contingency
chi2_table=pd.DataFrame(columns=["feature","chi2 score","Null hypothesis","dependent"])
for i in catx:
    chi_score=chi2_contingency(pd.crosstab(df1[i],y))[1]
    alpha=0.05
    if chi_score<alpha:
        chi2_table.loc[i]=[i,chi_score,"Reject","Correlated"]
    else:
        chi2_table.loc[i]=[i,chi_score,"Accept","Not correlated"]
chi2_table 
    

,feature,chi2 score,Null hypothesis,dependent
channelname,channelname,6.012658e-37,Reject,Correlated
category,category,3.493555e-150,Reject,Correlated
subcategory,subcategory,0.000000e+00,Reject,Correlated
issuereportedat,issuereportedat,9.758028e-01,Accept,Not correlated
issueresponded,issueresponded,5.035384e-01,Accept,Not correlated
surveyresponsedate,surveyresponsedate,5.986044e-11,Reject,Correlated
agentname,agentname,1.366200e-258,Reject,Correlated
supervisor,supervisor,2.501363e-77,Reject,Correlated
manager,manager,1.599412e-65,Reject,Correlated
tenurebucket,tenurebucket,6.486032e-46,Reject,Correlated


In [17]:
def cramersV(x,y):
    cm=pd.crosstab(x,y)
    chi2=chi2_contingency(cm)[0]
    n=cm.sum().sum()
    r,k=cm.shape
    return np.sqrt(chi2/(n*(min(r,k)-1)))

for i in catx:
    print(i,"    |    ",cramersV(df1[i],y))


channelname     |     0.034573533762500476
category     |     0.05168095341149951
subcategory     |     0.09640173716158151
issuereportedat     |     0.6069062152526554
issueresponded     |     0.612188083644408
surveyresponsedate     |     0.02789183022718758
agentname     |     0.17616376172759612
supervisor     |     0.04828130104916926
manager     |     0.033907187523995325
tenurebucket     |     0.02851074323488494
agentshift     |     0.023843183682772013
responsehourscategory     |     0.07153808328294865


In [18]:
#Higher mi value higher non-linear correlation
from sklearn.feature_selection import mutual_info_classif
for i in numx:
    mi=mutual_info_classif(x[[i]],y)
    print(f"{i}    |  {mi}")

sentiment_score    |  [0.0573952]
issuemonth    |  [0.00456888]
issuerespondedmonth    |  [0.00177749]
responsehours    |  [0.0224477]


In [19]:
# Here we remove the issuereportedat, issueresponded based on cramersV and chi2_contigency
# Here we only select agentname as they are hierarchial with sy=upervisor and manager so we select lowet level as it tells about the upper levels. 
# Here we do drop the responsehours feature based on the mutual_classif_info 
x=x.drop(["issuereportedat","issueresponded","surveyresponsedate","supervisor","manager","responsehours"],axis=1)
x.head(2)

,channelname,category,subcategory,agentname,tenurebucket,agentshift,sentiment_score,issuemonth,issuerespondedmonth,responsehourscategory
0,Outcall,Product Queries,Life Insurance,Richard Buchanan,On Job Training,Morning,0.0,1.0,1.0,within 6
1,Outcall,Product Queries,Product Specific Information,Vicki Collins,>90,Morning,0.0,1.0,1.0,within 6


In [20]:
for i in x.columns:
    print(i,x[i].nunique())

channelname 3
category 12
subcategory 57
agentname 1371
tenurebucket 5
agentshift 5
sentiment_score 1252
issuemonth 12
issuerespondedmonth 12
responsehourscategory 5


In [21]:
from sklearn.preprocessing import OneHotEncoder
from category_encoders import BinaryEncoder
bnes=[]
ohes=[]
for i in x.columns:
    if x[i].nunique()>5:
        bnes.append(i)
    else:
        ohes.append(i)
bnes,ohes

(['category',
  'subcategory',
  'agentname',
  'sentiment_score',
  'issuemonth',
  'issuerespondedmonth'],
 ['channelname', 'tenurebucket', 'agentshift', 'responsehourscategory'])

In [22]:
from sklearn.compose import ColumnTransformer
ct=ColumnTransformer(
    [
        ("ohe",OneHotEncoder(),ohes),
        ("bnes",BinaryEncoder(),bnes),
    ],
    remainder="drop"
)

In [23]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,stratify=y)
x_train.shape,y_train.shape,x_test.shape,y_test.shape

((63790, 10), (63790,), (15948, 10), (15948,))

In [24]:
x_train=ct.fit_transform(x_train)
x_test=ct.transform(x_test)
x_train[0],x_test[0]

(array([0., 1., 0., 0., 1., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0.,
        1., 0., 0., 0., 1., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 1., 0., 8., 8.]),
 array([0.  , 1.  , 0.  , 0.  , 0.  , 0.  , 0.  , 1.  , 0.  , 0.  , 1.  ,
        0.  , 0.  , 0.  , 0.  , 0.  , 1.  , 0.  , 0.  , 0.  , 0.  , 1.  ,
        0.  , 0.  , 0.  , 0.  , 1.  , 1.  , 0.  , 1.  , 0.  , 1.  , 0.  ,
        0.  , 0.  , 0.  , 0.  , 1.  , 0.  , 0.66, 8.  , 8.  ]))

In [25]:
x_train.shape

(63790, 42)

In [26]:
index_remove=[0,3,8,13]
cols=[col for col in range(x_train.shape[1]) if col not in index_remove]
x_train=x_train[:,cols]
x_test=x_test[:,cols]

In [27]:
x_train.shape,x_test.shape

((63790, 38), (15948, 38))

In [28]:
class Data(Dataset):
    def __init__(self,x,y):
        # x.issuereportedat=x.issuereportedat.astype("datetime64[ns]")
        # x.issueresponded=x.issueresponded.astype("datetime64[ns]")
        # x.surveyresponsedate=x.surveyresponsedate.astype("datetime64[ns]")
        # x["responsehours"]=(x.issueresponded-x.issuereportedat).dt.total_seconds()/3600
        # x["responsehourscategory"]=pd.cut(x.responsehours,bins=[0,6,12,18,24,3000],labels=["within 6","6-12","12-18","18-24","24+"])
        # x=x.drop(["issuereportedat","issueresponded","surveyresponsedate","supervisor","manager","responsehours"],axis=1)
        # ct=ColumnTransformer(
        #     [
        #         ("ohe",OneHotEncoder(),ohes),
        #         ("bnes",BinaryEncoder(),bnes),
        #     ],
        #     remainder="passthrough"
        # )
        self.x=torch.tensor(x,dtype=torch.float32)
        self.y=torch.tensor(y.values,dtype=torch.float32)
        
    def __getitem__(self,index):
        return self.x[index],self.y[index]
    
    def __len__(self):
        return self.x.shape[0]

In [29]:
train_data=Data(x_train,y_train)
len(train_data)

63790

In [30]:
test_data=Data(x_test,y_test)
len(test_data)

15948

In [31]:
test_data[0][0].shape

torch.Size([38])

In [32]:
train_dataloader=DataLoader(
    dataset=train_data,
    batch_size=16,
    drop_last=False,
    shuffle=True
)
test_dataloader=DataLoader(
    dataset=test_data,
    shuffle=False,
    drop_last=True,
    batch_size=16,
)

In [33]:
for batch,(x,y) in enumerate(train_dataloader):
    print(batch,x.shape,y.shape)
    break

0 torch.Size([16, 38]) torch.Size([16])


In [34]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc0=nn.Linear(
            in_features=38,
            out_features=64,
        )
        self.fc1=nn.Linear(
            in_features=64,
            out_features=128,
        )
        self.fc2=nn.Linear(
            in_features=128,
            out_features=64,
        )
        self.fc3=nn.Linear(
            in_features=64,
            out_features=32,
        )
        self.fc4=nn.Linear(
            in_features=32,
            out_features=1,
        )
        
    def forward(self,x):
        x=nn.functional.relu(self.fc0(x))
        x=nn.functional.relu(self.fc1(x))
        x=nn.functional.relu(self.fc2(x))
        x=nn.functional.relu(self.fc3(x))
        x=self.fc4(x)
        return x 

In [35]:
model=NeuralNetwork()
model

NeuralNetwork(
  (fc0): Linear(in_features=38, out_features=64, bias=True)
  (fc1): Linear(in_features=64, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=64, bias=True)
  (fc3): Linear(in_features=64, out_features=32, bias=True)
  (fc4): Linear(in_features=32, out_features=1, bias=True)
)

In [36]:
for params in model.parameters():
    print(params)
    break

Parameter containing:
tensor([[-3.1879e-02, -1.2637e-01, -6.6692e-02,  ...,  1.3313e-01,
         -1.9805e-02, -4.9363e-02],
        [-1.2350e-01, -1.4129e-01,  1.4708e-01,  ...,  9.9624e-02,
          5.3738e-02,  3.2231e-05],
        [-1.5495e-01,  1.6073e-01,  1.5848e-01,  ..., -1.0489e-01,
          1.1145e-01,  8.3984e-02],
        ...,
        [-1.4169e-01,  1.2876e-01, -1.4883e-01,  ...,  2.2890e-02,
          1.0890e-01,  1.5607e-01],
        [-5.0267e-02,  6.3434e-03, -2.8286e-02,  ..., -1.4751e-01,
          3.1726e-02, -1.5314e-01],
        [ 1.4058e-01, -1.1670e-01, -1.7283e-02,  ..., -9.6435e-02,
          6.8460e-02, -2.3935e-02]], requires_grad=True)


In [37]:
import warnings
warnings.filterwarnings("ignore")

In [38]:
import torchinfo
from torchinfo import summary
summary(model,input_size=(16,38))

Layer (type:depth-idx)                   Output Shape              Param #
NeuralNetwork                            [16, 1]                   --
├─Linear: 1-1                            [16, 64]                  2,496
├─Linear: 1-2                            [16, 128]                 8,320
├─Linear: 1-3                            [16, 64]                  8,256
├─Linear: 1-4                            [16, 32]                  2,080
├─Linear: 1-5                            [16, 1]                   33
Total params: 21,185
Trainable params: 21,185
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.34
Input size (MB): 0.00
Forward/backward pass size (MB): 0.04
Params size (MB): 0.08
Estimated Total Size (MB): 0.12

In [39]:
device=torch.device("cpu")

In [40]:
loss_fn=nn.L1Loss()
optimizer=optim.Adam(model.parameters(),lr=0.001)

In [41]:
y_test.min(),y_test.max()

(np.int64(1), np.int64(5))

In [42]:
def predict():
    model.eval()
    running_test_loss = 0.0
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for x, y in test_dataloader:
            x = x.float().to(device)
            y = y.float().unsqueeze(1).to(device)

            pred = model(x)
            loss = loss_fn(pred, y)

            running_test_loss += loss.item()

            pred = torch.clamp(pred, 1, 5)
            pred_round = torch.round(pred)

            all_preds.extend(pred_round.cpu().numpy().flatten())
            all_targets.extend(y.cpu().numpy().flatten())

    avg_test_loss = running_test_loss / len(test_dataloader)

    all_preds = np.array(all_preds)
    all_targets = np.array(all_targets)

    test_acc = (all_preds == all_targets).mean() * 100

    return avg_test_loss, test_acc, all_preds, all_targets

In [43]:
epochs = 20
trainloss = []
trainaccs = []
testlosses = []
testaccs = []

model.to(device)

for i in range(epochs):
    model.train()
    running_train_loss = 0.0
    total, correct = 0, 0

    for x, y in tqdm(train_dataloader, total=len(train_dataloader)):
        x = x.float().to(device)
        y = y.float().unsqueeze(1).to(device)

        optimizer.zero_grad()

        pred = model(x)
        loss = loss_fn(pred, y)

        running_train_loss += loss.item()

        loss.backward()
        optimizer.step()

        pred = torch.clamp(pred, 1, 5)
        pred_round = torch.round(pred)

        total += y.size(0)
        correct += (pred_round == y).sum().item()

    avg_train_loss = running_train_loss / len(train_dataloader)
    train_acc = correct / total * 100

    test_loss, test_acc, y_pred, y_actual = predict()

    trainaccs.append(train_acc)
    trainloss.append(avg_train_loss)
    testaccs.append(test_acc)
    testlosses.append(test_loss)

    print(
        f"Epoch: {i+1}/{epochs}\t| "
        f"Train loss: {avg_train_loss:.4f}\t| "
        f"Test loss: {test_loss:.4f}\t| "
        f"Train acc: {train_acc:.4f}\t| "
        f"Test acc: {test_acc:.4f}"
    )

100%|██████████| 3987/3987 [00:08<00:00, 491.29it/s]


Epoch: 1/20	| Train loss: 0.8667	| Test loss: 0.8263	| Train acc: 63.4378	| Test acc: 67.3318


100%|██████████| 3987/3987 [00:09<00:00, 405.36it/s]


Epoch: 2/20	| Train loss: 0.7655	| Test loss: 0.7197	| Train acc: 67.8225	| Test acc: 69.4403


100%|██████████| 3987/3987 [00:17<00:00, 232.44it/s]


...

Epoch: 18/20	| Train loss: 0.7036	| Test loss: 0.7016	| Train acc: 70.5957	| Test acc: 69.4152


100%|██████████| 3987/3987 [00:09<00:00, 416.78it/s]


Epoch: 19/20	| Train loss: 0.7033	| Test loss: 0.7006	| Train acc: 70.6035	| Test acc: 70.7016


100%|██████████| 3987/3987 [00:14<00:00, 284.03it/s]


Epoch: 20/20	| Train loss: 0.7015	| Test loss: 0.6973	| Train acc: 70.6224	| Test acc: 70.7392


In [44]:
test_loss, test_acc, y_pred, y_actual = predict()

from sklearn.metrics import mean_absolute_error, mean_squared_error

print("Rounded Accuracy:", test_acc)
print("MAE:", mean_absolute_error(y_actual, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_actual, y_pred)))
print("Within ±1:", (np.abs(y_actual - y_pred) <= 1).mean() * 100)

Rounded Accuracy: 70.73920682730925
MAE: 0.6889432668685913
RMSE: 1.4741965417043272
Within ±1: 84.01731927710844


In [ ]:
import joblib
# joblib.dump(model.state_dict(),"model_weights.pkl")

['model_weights.pkl']

In [47]:
# joblib.load("model_weights.pkl")

In [ ]:
def predict():
    sample = df1.loc[11].drop([
        "csatscore",
        "issuereportedat",
        "issueresponded",
        "surveyresponsedate",
        "supervisor",
        "manager",
        "responsehours"
    ]).to_frame().T

    sample = ct.transform(sample)
    cols=[col for col in range(sample.shape[1]) if col not in index_remove]
    sample=sample[:,cols]
    sample = torch.tensor(np.array(sample, dtype=np.float32), dtype=torch.float32)
    finalmodel=NeuralNetwork()
    finalmodel.load_state_dict(joblib.load("model_weights.pkl"))
    finalmodel.eval()
    with torch.no_grad():
        finalmodel.to(device)
        output=finalmodel(sample)
    return np.round(output.item())

In [63]:
predict()

np.float64(1.0)